In [1]:
import sys
sys.path.append("..")
from functools import partial
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.chunking import clause_chunks
from src.retrieval import make_dense_ranker
from src.bm25 import make_bm25_ranker
from src.queries import CLAUSE_KEYWORDS

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)
chunker = partial(clause_chunks, max_size=1500)
tag = "clause-200-1500-200"
dense_q = make_dense_ranker(chunker, tag=tag)
dense_kw = make_dense_ranker(chunker, tag=tag, queries=CLAUSE_KEYWORDS)
bm25 = make_bm25_ranker(chunker)

rows = []
for cl in TARGET_CLAUSES:
    for i in dev_idx:
        c = contracts[i]
        rows.append({
            "clause": cl,
            "has_clause": len(c["gold"][cl]) > 0,
            "dense_q": dense_q(c, cl)[0]["score"],
            "dense_kw": dense_kw(c, cl)[0]["score"],
            "bm25": bm25(c, cl)[0]["score"],
        })
feat = pd.DataFrame(rows)
feat.to_csv("../eval/abstention_features_dev.csv", index=False)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
out = []
for cl in TARGET_CLAUSES:
    d = feat[feat.clause == cl]
    out.append({
        "clause": cl,
        "present": int(d.has_clause.sum()),
        "absent": int((~d.has_clause).sum()),
        **{s: round(roc_auc_score(d.has_clause, d[s]), 3) for s in ["dense_q", "dense_kw", "bm25"]},
    })
pd.DataFrame(out)

,clause,present,absent,dense_q,dense_kw,bm25
0,Governing Law,306,51,0.933,0.954,0.985
1,Renewal Term,118,239,0.815,0.849,0.879
2,Notice Period To Terminate Renewal,71,286,0.776,0.757,0.792
3,Termination For Convenience,118,239,0.653,0.682,0.722
4,Non-Compete,75,282,0.801,0.814,0.815
5,Cap On Liability,190,167,0.891,0.888,0.932
6,Uncapped Liability,81,276,0.829,0.855,0.785


In [3]:
def at_recall(feat, signal, target=0.90):
    out = []
    for cl in TARGET_CLAUSES:
        d = feat[feat.clause == cl]
        pos = d[d.has_clause][signal]
        absent = d[~d.has_clause][signal]
        t = np.quantile(pos, 1 - target)
        out.append({
            "clause": cl,
            "threshold": round(float(t), 3),
            "found_when_present": round(float((pos >= t).mean()), 2),
            "false_alarm_rate": round(float((absent >= t).mean()), 2),
        })
    return pd.DataFrame(out)

at_recall(feat, "dense_kw")

,clause,threshold,found_when_present,false_alarm_rate
0,Governing Law,0.614,0.90,0.16
1,Renewal Term,0.617,0.90,0.52
2,Notice Period To Terminate Renewal,0.662,0.90,0.57
3,Termination For Convenience,0.698,0.90,0.59
4,Non-Compete,0.627,0.89,0.62
5,Cap On Liability,0.660,0.90,0.37
6,Uncapped Liability,0.683,0.90,0.37
